In [ ]:
# Loading libraries
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

import os
from pathlib import Path

# Automatically get the current working directory
current_dir = Path(os.getcwd())

# Define file paths using pathlib
train_file = current_dir / "input" / "train.csv"
test_file = current_dir / "input" / "test.csv"
sample_sub_file = current_dir / "input" / "sample_submission.csv"

# Check if path exists
if train_file.exists() and test_file.exists() and sample_sub_file.exists():
    print("All files exist and path is correctly set.")
else:
    print("Some files are missing or path is not correctly set.")

All files exist and path is correctly set.


In [15]:
train = pd.read_csv(train_file)
test = pd.read_csv(test_file)
sample_sub = pd.read_csv(sample_sub_file)

In [17]:
# Numerization（Label Encoding）
train["NAME_CONTRACT_TYPE"].replace({'Cash loans': 0, 'Revolving loans': 1}, inplace=True)
test["NAME_CONTRACT_TYPE"].replace({'Cash loans': 0, 'Revolving loans': 1}, inplace=True)

train["CODE_GENDER"].replace({'F': 0, 'M': 1, 'XNA': np.nan}, inplace=True)
test["CODE_GENDER"].replace({'F': 0, 'M': 1, 'XNA': np.nan}, inplace=True)

In [18]:
# Numerization of ORGANIZATION_TYPE (Count Encoding）
organization_ce = train["ORGANIZATION_TYPE"].value_counts()
train["ORGANIZATION_TYPE"] = train["ORGANIZATION_TYPE"].map(organization_ce)
test["ORGANIZATION_TYPE"] = test["ORGANIZATION_TYPE"].map(organization_ce)

In [19]:
# Complete missing values with the average
for col in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "CNT_FAM_MEMBERS"]:
    train[col].fillna(train[col].mean(), inplace=True)
    test[col].fillna(train[col].mean(), inplace=True)

In [20]:
# Complete missing values with 0
for col in ["AMT_REQ_CREDIT_BUREAU_HOUR", "AMT_REQ_CREDIT_BUREAU_MON", "AMT_REQ_CREDIT_BUREAU_QRT", "AMT_REQ_CREDIT_BUREAU_YEAR"]:
    train[col].fillna(0, inplace=True)
    test[col].fillna(0, inplace=True)

In [21]:
# Treat values above 60 (outliers) in OWN_CAR_AGE as missing values.
train.loc[train["OWN_CAR_AGE"] >= 60, "OWN_CAR_AGE"] = np.nan
test.loc[test["OWN_CAR_AGE"] >= 60, "OWN_CAR_AGE"] = np.nan

# Divide OWN_CAR_AGE into groups
train["OWN_CAR_AGE"] = train["OWN_CAR_AGE"] // 10
test["OWN_CAR_AGE"] = test["OWN_CAR_AGE"] // 10

# Apply One Hot Encoding to OWN_CAR_AGE
train_car_age_ohe = pd.get_dummies(train["OWN_CAR_AGE"]).add_prefix("OWN_CAR_AGE_")
test_car_age_ohe = pd.get_dummies(test["OWN_CAR_AGE"]).add_prefix("OWN_CAR_AGE_")

train = pd.concat([train, train_car_age_ohe], axis=1)
test = pd.concat([test, test_car_age_ohe], axis=1)

train.drop('OWN_CAR_AGE', axis=1, inplace=True)
test.drop('OWN_CAR_AGE', axis=1, inplace=True)

In [ ]:
from typing import Tuple, List

def add_effective_features(train: pd.DataFrame, test: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    tr, te = train.copy(), test.copy()
    new_cols: List[str] = []

    def add_feature(name: str, tr_v: pd.Series, te_v: pd.Series):
        tr[name] = tr_v
        te[name] = te_v
        new_cols.append(name)

    # 1) LOG_AMT_INCOME_TOTAL
    if 'AMT_INCOME_TOTAL' in tr.columns:
        tr_raw = np.log1p(tr['AMT_INCOME_TOTAL'])
        te_raw = np.log1p(te['AMT_INCOME_TOTAL'])
        add_feature('LOG_AMT_INCOME_TOTAL', tr_raw, te_raw)

    # 2) EXT_MEAN
    ext_cols = [c for c in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'] if c in tr.columns]
    if ext_cols:
        # Row-wise statistical aggregations
        tr_raw = tr[ext_cols].mean(axis=1, skipna=True)
        te_raw = te[ext_cols].mean(axis=1, skipna=True)
        add_feature('EXT_MEAN', tr_raw, te_raw)


    # 3) ANNUITY_INCOME_RATIO = AMT_ANNUITY / AMT_INCOME_TOTAL
    if {'AMT_ANNUITY', 'AMT_INCOME_TOTAL'}.issubset(tr.columns):
        tr_raw = tr['AMT_ANNUITY'] / tr['AMT_INCOME_TOTAL'].where(tr['AMT_INCOME_TOTAL'] > 0, np.nan)
        te_raw = te['AMT_ANNUITY'] / te['AMT_INCOME_TOTAL'].where(te['AMT_INCOME_TOTAL'] > 0, np.nan)
        add_feature('ANNUITY_INCOME_RATIO', tr_raw, te_raw)

    # 4) CREDIT_TERM = AMT_CREDIT / AMT_ANNUITY
    if {'AMT_CREDIT', 'AMT_ANNUITY'}.issubset(tr.columns):
        term_tr = tr['AMT_CREDIT'] / tr['AMT_ANNUITY'].where(tr['AMT_ANNUITY'] > 0, np.nan)
        term_te = te['AMT_CREDIT'] / te['AMT_ANNUITY'].where(te['AMT_ANNUITY'] > 0, np.nan)
        add_feature('CREDIT_TERM', term_tr, term_te)

    # 5) OCCUP_X_ORG = OCCUPATION_TYPE + ORGANIZATION_TYPE
    occ_org_tr = tr['OCCUPATION_TYPE'].astype(str) + "_" + tr['ORGANIZATION_TYPE'].astype(str)
    occ_org_te = te['OCCUPATION_TYPE'].astype(str) + "_" + te['ORGANIZATION_TYPE'].astype(str)
    add_feature('OCCUP_X_ORG', occ_org_tr, occ_org_te)

    return tr, te, new_cols

# Apply feature engineering
train, test, _new_cols = add_effective_features(train, test)

In [ ]:
# Drop target and ID, align to the columns common to train and test
num_cols_train = train.drop(columns=['TARGET', 'SK_ID_CURR']).columns
num_cols_test  = test.columns
num_cols = num_cols_train.intersection(num_cols_test)

# Build arrays using numeric columns only (ensure the same column order for train/test) 
X = train[num_cols]
y = train['TARGET']
X_test = test[num_cols]

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

# ==========================================
# 1. SETUP STRATIFIED K-FOLD
# ==========================================
N_SPLITS = 5
kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=0)

# Arrays to store Out-Of-Fold (OOF) and Test predictions for each model
lgb_oof = np.zeros(len(X))
lgb_test = np.zeros(len(X_test))

xgb_oof = np.zeros(len(X))
xgb_test = np.zeros(len(X_test))

cat_oof = np.zeros(len(X))
cat_test = np.zeros(len(X_test))

# Arrays to store importances for each model
lgb_importances = np.zeros(X.shape[1])
xgb_importances = np.zeros(X.shape[1])
cat_importances = np.zeros(X.shape[1])

# ==========================================
# 2. MODEL TRAINING LOOP
# ==========================================
print("Starting K-Fold Cross-Validation Training...\n")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"### Fold {fold + 1} / {N_SPLITS} ---")
    
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    X_test_fold = X_test.copy()
    
    # ------------------------------------------
    # TARGET ENCODING USING SKLEARN
    # ------------------------------------------
    # Initialize the encoder
    encoder = TargetEncoder(smooth="auto", target_type="binary", random_state=0)
    target_cols = ["NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "OCCUPATION_TYPE", 'OCCUP_X_ORG']

    # Fit and transform training data (prevents overfitting via internal CV)
    X_train[target_cols] = encoder.fit_transform(X_train[target_cols], y_train)
    
    # Transform validation and test sets safely using the fitted encoder
    X_val[target_cols] = encoder.transform(X_val[target_cols])
    X_test_fold[target_cols] = encoder.transform(X_test_fold[target_cols])

    # ------------------------------------------
    # A. LightGBM
    # ------------------------------------------
    lgb_model = lgb.LGBMClassifier(
        objective='binary',
        metric='auc',
        boosting_type='gbdt',
        learning_rate=0.05603118738468992,
        num_leaves=42,
        max_depth=3,
        min_child_samples=327,
        subsample=0.7318275133836804,
        colsample_bytree=0.38397477075501324,
        reg_alpha=9.010534515746432,
        reg_lambda=0.006051147817228469,
        n_estimators=2000,
        random_state=0,
        verbose=-1
    )

    lgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )
    lgb_importances += lgb_model.feature_importances_ / N_SPLITS
    lgb_oof[val_idx] = lgb_model.predict_proba(X_val)[:, 1]
    lgb_test += lgb_model.predict_proba(X_test_fold)[:, 1] / N_SPLITS
    lgb_probs = lgb_model.predict_proba(X_val)[:, 1]
    lgb_fold_auc = roc_auc_score(y_val, lgb_probs)

    # ------------------------------------------
    # B. XGBoost
    # ------------------------------------------
    xgb_model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        learning_rate=0.008034344475619404,
        max_depth=7,
        n_estimators=2000,
        min_child_weight=15,
        subsample=0.9010984903770198,
        colsample_bytree=0.35218545057583955, 
        gamma=0.7854083114461319,
        reg_alpha=1.2273800987852967,
        reg_lambda=0.0062353771356731605,
        random_state=0,
        tree_method='hist',
        early_stopping_rounds=100
    )
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    xgb_importances += xgb_model.feature_importances_ / N_SPLITS
    xgb_probs = xgb_model.predict_proba(X_val)[:, 1]
    xgb_oof[val_idx] = xgb_model.predict_proba(X_val)[:, 1]
    xgb_test += xgb_model.predict_proba(X_test_fold)[:, 1] / N_SPLITS
    xgb_fold_auc = roc_auc_score(y_val, xgb_probs)

    # ------------------------------------------
    # C. CatBoost
    # ------------------------------------------
    cat_model = CatBoostClassifier(
        loss_function='Logloss',
        eval_metric='AUC',
        learning_rate=0.03400372129740376,
        depth=6,
        l2_leaf_reg=6.007249475906204,
        iterations=2000,
        random_seed=0,
        verbose=False
    )
    cat_model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100
    )
    cat_importances += cat_model.get_feature_importance() / N_SPLITS
    cat_oof[val_idx] = cat_model.predict_proba(X_val)[:, 1]
    cat_test += cat_model.predict_proba(X_test_fold)[:, 1] / N_SPLITS
    cat_probs = cat_model.predict_proba(X_val)[:, 1]
    cat_fold_auc = roc_auc_score(y_val, cat_probs)

    # Print Fold-level ROC-AUC Scores
    print(f"- LightGBM Fold ROC-AUC : {lgb_fold_auc:.5f}")
    print(f"- XGBoost Fold ROC-AUC  : {xgb_fold_auc:.5f}")
    print(f"- CatBoost Fold ROC-AUC : {cat_fold_auc:.5f}\n")

# ==========================================
# 3. EVALUATE INDIVIDUAL MODELS
# ==========================================
auc_lgb = roc_auc_score(y, lgb_oof)
auc_xgb = roc_auc_score(y, xgb_oof)
auc_cat = roc_auc_score(y, cat_oof)

print("\n==========================================")
print("### INDIVIDUAL OOF ROC-AUC SCORES:")
print(f"- LightGBM AUC : {auc_lgb:.5f}")
print(f"- XGBoost AUC  : {auc_xgb:.5f}")
print(f"- CatBoost AUC : {auc_cat:.5f}")

# ==========================================
# 4. BLEND / ENSEMBLE PREDICTIONS
# ==========================================
w_lgb, w_xgb, w_cat = 0.4, 0.3, 0.3 

ensemble_oof = (w_lgb * lgb_oof) + (w_xgb * xgb_oof) + (w_cat * cat_oof)
ensemble_test = (w_lgb * lgb_test) + (w_xgb * xgb_test) + (w_cat * cat_test)

print("------------------------------------------")
print(f"Ensemble OOF ROC-AUC Score: {roc_auc_score(y, ensemble_oof):.5f}")
print("==========================================")

# Takes ~10mins to complete

Starting K-Fold Cross-Validation Training...

### Fold 1 / 5 ---
- LightGBM Fold ROC-AUC : 0.76328
- XGBoost Fold ROC-AUC  : 0.76287
- CatBoost Fold ROC-AUC : 0.76087

### Fold 2 / 5 ---
- LightGBM Fold ROC-AUC : 0.76469
- XGBoost Fold ROC-AUC  : 0.76358
- CatBoost Fold ROC-AUC : 0.76139

### Fold 3 / 5 ---
- LightGBM Fold ROC-AUC : 0.76123
- XGBoost Fold ROC-AUC  : 0.75969
- CatBoost Fold ROC-AUC : 0.75953

### Fold 4 / 5 ---
- LightGBM Fold ROC-AUC : 0.76159
- XGBoost Fold ROC-AUC  : 0.76084
- CatBoost Fold ROC-AUC : 0.75928

### Fold 5 / 5 ---
- LightGBM Fold ROC-AUC : 0.76308
- XGBoost Fold ROC-AUC  : 0.76200
- CatBoost Fold ROC-AUC : 0.76077


### INDIVIDUAL OOF ROC-AUC SCORES:
- LightGBM AUC : 0.76278
- XGBoost AUC  : 0.76178
- CatBoost AUC : 0.76035
------------------------------------------
Ensemble OOF ROC-AUC Score: 0.76317


In [ ]:
# Put the prediction into the format of submission
sample_sub['TARGET'] = ensemble_test
sample_sub

# Create the "output" directory if it doesn't exist
output_dir = current_dir / "output"
os.makedirs(output_dir, exist_ok=True)

# Specify the new output file path
output_file = output_dir / "submission.csv"

# Save the CSV file to the "output" directory
sample_sub.to_csv(output_file, index=False)

In [4]:
import zipfile

# Define the file names
notebook_filename = "tutorial.ipynb" # Enter the name of this notebook
zip_filename = "tutorial.zip"

# Define the full paths using current_dir
notebook_path = current_dir / notebook_filename
zip_output_path = current_dir / zip_filename

# Check if the notebook file exists and create the ZIP file
if notebook_path.exists():
    with zipfile.ZipFile(zip_output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Add the notebook to the ZIP file (arcname avoids including the full folder structure)
        zipf.write(notebook_path, arcname=notebook_filename)
    print(f"Successfully created. ")
else:
    print(f"Error: Could not find '{notebook_filename}' in {current_dir}.")
    print("Please check if the notebook name and the current directory are correct.")

Successfully created. 
